In [ ]:
"""
21_isolation_forest.py
==================================================================

ISOLATION FOREST

Dataset:
Wine Dataset

REAL WORLD WORKFLOW

1. Data Understanding
2. EDA
3. Feature Selection
4. Scaling
5. Baseline Isolation Forest
6. Anomaly Detection
7. Anomaly Scoring
8. Validation
9. Hyperparameter Tuning
10. PCA Visualization
11. Anomaly Profiling
12. Contamination Analysis
13. Stability Analysis
14. Deployment Readiness
15. Interview Questions

==================================================================
"""

# ==========================================================
# STEP 0 : IMPORTS
# ==========================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine

from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import IsolationForest

from sklearn.decomposition import PCA

import joblib


# ==========================================================
# STEP 1 : DATA UNDERSTANDING
# ==========================================================

wine = load_wine()

df = pd.DataFrame(
    wine.data,
    columns=wine.feature_names
)

print("\nHEAD")
print(df.head())

print("\nINFO")
print(df.info())

print("\nDESCRIBE")
print(df.describe())

print("\nSHAPE")
print(df.shape)


# ==========================================================
# STEP 2 : EDA
# ==========================================================

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicates")
print(df.duplicated().sum())

plt.figure(figsize=(12,8))

sns.heatmap(
    df.corr(),
    cmap="coolwarm"
)

plt.title("Correlation Matrix")

plt.show()


# ==========================================================
# STEP 3 : FEATURE SELECTION
# ==========================================================

X = df.copy()

# ==========================================================
# STEP 4 : SCALING
# ==========================================================

"""
Isolation Forest is tree based.

Scaling is NOT mandatory.

However, keeping a consistent
pipeline is usually preferred.
"""

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

# ==========================================================
# STEP 5 : BASELINE MODEL
# ==========================================================

iso = IsolationForest(

    n_estimators=200,

    contamination=0.05,

    max_samples='auto',

    bootstrap=False,

    random_state=42,

    n_jobs=-1

)

"""
IMPORTANT PARAMETERS

n_estimators
------------
Number of trees

contamination
-------------
Expected anomaly percentage

max_samples
-----------
Samples per tree

bootstrap
----------
Sampling with replacement

n_jobs
-------
Parallel execution
"""

# ==========================================================
# STEP 6 : ANOMALY DETECTION
# ==========================================================

labels = iso.fit_predict(
    X_scaled
)

"""
OUTPUT

1  -> Normal

-1 -> Anomaly
"""

df["anomaly"] = labels

print("\nAnomaly Counts")

print(
    df["anomaly"].value_counts()
)

# ==========================================================
# STEP 7 : ANOMALY SCORING
# ==========================================================

"""
Lower score

=> more anomalous
"""

scores = iso.decision_function(
    X_scaled
)

df["anomaly_score"] = scores

print("\nLowest Scores")

print(

    df.sort_values(
        "anomaly_score"
    )

    .head(10)

)

# ==========================================================
# STEP 8 : VALIDATION
# ==========================================================

"""
Unsupervised Validation

No target available.

Validation focuses on:

1. Number of anomalies
2. Score distribution
3. Business review
"""

anomaly_count = np.sum(
    labels == -1
)

anomaly_percent = (

    anomaly_count
    /
    len(df)

) * 100

print("\nAnomaly Count")

print(anomaly_count)

print("\nAnomaly Percent")

print(round(
    anomaly_percent,
    2
), "%")

# ==========================================================
# STEP 9 : HYPERPARAMETER TUNING
# ==========================================================

results = []

for contamination in [

    0.01,
    0.03,
    0.05,
    0.10

]:

    model = IsolationForest(

        contamination=contamination,

        random_state=42

    )

    lbl = model.fit_predict(
        X_scaled
    )

    anomalies = np.sum(
        lbl == -1
    )

    results.append([

        contamination,

        anomalies

    ])

tuning_df = pd.DataFrame(

    results,

    columns=[

        "contamination",

        "anomaly_count"

    ]

)

print("\nContamination Study")

print(tuning_df)

# ==========================================================
# STEP 10 : PCA VISUALIZATION
# ==========================================================

pca = PCA(
    n_components=2
)

X_pca = pca.fit_transform(
    X_scaled
)

plt.figure(figsize=(10,6))

plt.scatter(

    X_pca[df["anomaly"]==1,0],

    X_pca[df["anomaly"]==1,1],

    label="Normal"

)

plt.scatter(

    X_pca[df["anomaly"]==-1,0],

    X_pca[df["anomaly"]==-1,1],

    label="Anomaly"

)

plt.title(
    "Isolation Forest Results"
)

plt.xlabel("PC1")

plt.ylabel("PC2")

plt.legend()

plt.show()

# ==========================================================
# STEP 11 : ANOMALY PROFILING
# ==========================================================

normal_data = df[
    df["anomaly"] == 1
]

anomaly_data = df[
    df["anomaly"] == -1
]

print("\nNormal Mean")

print(
    normal_data.mean(
        numeric_only=True
    )
)

print("\nAnomaly Mean")

print(
    anomaly_data.mean(
        numeric_only=True
    )
)

"""
Very important in industry.

Goal:

Understand WHY
these records are anomalous.
"""

# ==========================================================
# STEP 12 : CONTAMINATION ANALYSIS
# ==========================================================

plt.figure(figsize=(8,5))

sns.histplot(

    df["anomaly_score"],

    kde=True

)

plt.title(
    "Anomaly Score Distribution"
)

plt.show()

# ==========================================================
# STEP 13 : STABILITY ANALYSIS
# ==========================================================

seeds = [

    1,
    42,
    100,
    500

]

for seed in seeds:

    model = IsolationForest(

        contamination=0.05,

        random_state=seed

    )

    lbl = model.fit_predict(
        X_scaled
    )

    anomalies = np.sum(
        lbl == -1
    )

    print(

        f"Seed={seed}"

        f" -> Anomalies={anomalies}"

    )

# ==========================================================
# STEP 14 : DEPLOYMENT READINESS
# ==========================================================

joblib.dump(

    iso,

    "isolation_forest.pkl"

)

joblib.dump(

    scaler,

    "isolation_scaler.pkl"

)

print("\nArtifacts Saved")

print("isolation_forest.pkl")

print("isolation_scaler.pkl")

# Example Prediction

sample = X.iloc[[0]]

sample_scaled = scaler.transform(
    sample
)

prediction = iso.predict(
    sample_scaled
)

score = iso.decision_function(
    sample_scaled
)

print("\nPrediction")

print(prediction)

print("\nAnomaly Score")

print(score)

# ==========================================================
# STEP 15 : INTERVIEW QUESTIONS
# ==========================================================

"""
1. What is anomaly detection?

2. What is Isolation Forest?

3. Why is it called Isolation Forest?

4. How does Isolation Forest work?

5. What does isolation mean?

6. Why anomalies get isolated faster?

7. What is contamination?

8. How do you choose contamination?

9. What is decision_function()?

10. What does negative score mean?

11. Why no target variable is needed?

12. Isolation Forest vs DBSCAN?

13. Isolation Forest vs One-Class SVM?

14. Isolation Forest vs LOF?

15. Why is Isolation Forest scalable?

16. Why tree-based methods help?

17. Is scaling mandatory?

18. How do you validate anomaly detection?

19. What are real-world applications?

20. Explain fraud detection using IF.

21. Explain network intrusion detection.

22. Why does IF work well in high dimensions?

23. What is max_samples?

24. What is n_estimators?

25. Explain Isolation Forest end-to-end.
"""

# ==========================================================
# INTERVIEW CHEAT SHEET
# ==========================================================

"""
NORMAL POINT

Needs many splits
to isolate.

ANOMALY

Needs very few splits
to isolate.

Hence:

Short Path Length
=
Anomaly

Long Path Length
=
Normal Observation

This is the entire
intuition behind
Isolation Forest.
"""

# ==========================================================
# END OF FILE
# ==========================================================